In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import re

from tensorflow.keras.layers import Dense, LSTM, Input, Dropout, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

with open('drive/MyDrive/Colab Notebooks/positive.txt', 'r', encoding='utf-8') as f:
    texts_true = f.readlines()
    texts_true[0] = texts_true[0].replace('\ufeff', '') #убираем первый невидимый символ

with open('drive/MyDrive/Colab Notebooks/negative.txt', 'r', encoding='utf-8') as f:
    texts_false = f.readlines()
    texts_false[0] = texts_false[0].replace('\ufeff', '') #убираем первый невидимый символ

texts_true = [t.strip() for t in texts_true if t.strip() != ""]
texts_false = [t.strip() for t in texts_false if t.strip() != ""]


texts = texts_true + texts_false
count_true = len(texts_true)
count_false = len(texts_false)
total_lines = count_true + count_false
print(count_true, count_false, total_lines)


maxWordsCount = 5000
tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»', lower=True, split=' ', char_level=False)
tokenizer.fit_on_texts(texts)

dist = list(tokenizer.word_counts.items())
print(dist[:10])
print(texts[0][:100])


max_text_len = 20
data = tokenizer.texts_to_sequences(texts)
data_pad = pad_sequences(data, maxlen=max_text_len)
print(data_pad)

print( list(tokenizer.word_index.items()) )


X = data_pad
Y = np.array([[1, 0]]*count_true + [[0, 1]]*count_false)
print(X.shape, Y.shape)

indeces = np.random.choice(X.shape[0], size=X.shape[0], replace=False)
X = X[indeces]
Y = Y[indeces]


model = Sequential()
model.add(Embedding(maxWordsCount, 128, input_length = max_text_len))
model.add(LSTM(128, return_sequences=True))
model.add(LSTM(64))
model.add(Dense(2, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer=Adam(0.0001))

history = model.fit(X, Y, batch_size=32, epochs=50)

reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))

def sequence_to_text(list_of_indices):
    words = [reverse_word_map.get(letter) for letter in list_of_indices]
    return(words)

t = "Сегодня была страшная и плохая погода".lower()
# t = "Я никогда не сяду за руль, это же смертельно опасно! ".lower()
data = tokenizer.texts_to_sequences([t])
data_pad = pad_sequences(data, maxlen=max_text_len)
print( sequence_to_text(data[0]) )

res = model.predict(data_pad)
print(res, np.argmax(res), sep='\n')


90 300 390
[('ты', 85), ('можешь', 24), ('изменить', 22), ('свою', 15), ('жизнь', 51), ('если', 42), ('каждый', 61), ('день', 24), ('выбираешь', 22), ('веру', 8)]
Ты можешь изменить свою жизнь, если каждый день выбираешь веру в себя и делаешь шаг вперед, зная что
[[  53  132    6 ...   24  325  682]
 [  13   83   47 ...  686   24  326]
 [   8   53  687 ...   24  689  690]
 ...
 [   0    0    0 ...   11  391 1437]
 [   0    0    0 ...   68   69  212]
 [   2    5   45 ...    4   61  328]]
[('что', 1), ('я', 2), ('и', 3), ('не', 4), ('всё', 5), ('в', 6), ('меня', 7), ('ты', 8), ('ни', 9), ('то', 10), ('мне', 11), ('бы', 12), ('каждый', 13), ('кажется', 14), ('когда', 15), ('как', 16), ('жизнь', 17), ('это', 18), ('думаю', 19), ('если', 20), ('на', 21), ('потому', 22), ('только', 23), ('к', 24), ('равно', 25), ('с', 26), ('ведь', 27), ('у', 28), ('уже', 29), ('чем', 30), ('а', 31), ('раз', 32), ('быть', 33), ('себя', 34), ('тебя', 35), ('вперед', 36), ('способен', 37), ('становится', 38), 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.7442 - loss: 0.6855
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7736 - loss: 0.6554
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7743 - loss: 0.6027
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7512 - loss: 0.5064
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7503 - loss: 0.4371
Epoch 6/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7615 - loss: 0.3369
Epoch 7/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8389 - loss: 0.2240
Epoch 8/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9726 - loss: 0.1510
Epoch 9/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.0891
Epoch 10/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0443
Epoch 11/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0329
Epoch 12/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 

In [8]:
test_sentences = [
    "Сегодня была прекрасная и спокойная погода",
    "Мне было очень приятно начать день с хороших мыслей",
    "Я чувствую уверенность и силы двигаться вперед",
    "Этот день подарил мне свет, тепло и спокойствие",
    "Я рад, что у меня получилось справиться со всем",
    "Мне нравится ощущать гармонию и внутренний свет",
    "Сегодня всё прошло легко и очень хорошо",

    "Сегодня был ужасный и тяжелый день",
    "Мне грустно и тяжело справляться с этим",
    "Все кажется опасным и неприятным",
]



true_labels = [1,1,1,1,1, 1,1,0,0,0]   # 1 = neg, 0 = pos  (match order above)

pred_labels = []
for sent in test_sentences:
    seq = tokenizer.texts_to_sequences([sent.lower()])
    pad = pad_sequences(seq, maxlen=max_text_len)
    res = model.predict(pad)[0]
    pred = np.argmax(res)
    pred_labels.append(pred)

# Calculate accuracy
correct = sum(1 for p, t in zip(pred_labels, true_labels) if p == t)
accuracy = correct / len(true_labels)

print("Predicted labels:", pred_labels)
print("True labels:     ", true_labels)
print(f"Accuracy: {accuracy*100:.1f}%")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Predicted labels: [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]
True labels:      [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
Accuracy: 70.0%
